# Module 8.3: Regression Testing & CI/CD for RAG Systems

**Duration**: 35 minutes  
**Level**: 2  
**Prerequisites**: M3 (Deployment), M8.1 (RAGAS Evaluation), M8.2 (A/B Testing)

This module teaches automated quality assurance for RAG systems using GitHub Actions, DVC, and pytest-benchmark.

## Section 1: Introduction & Problem Statement

### The Horror Story

**Scenario**: You deploy three "small" changes:
1. Upgrade embedding model from `text-embedding-ada-002` to `text-embedding-3-small`
2. Tweak prompt formatting (add `\n\n` between instructions)
3. Adjust reranking threshold from 0.7 to 0.65

Each change seems minor. But together?

**Result**: Answer quality drops 40%. Discovered **three days later** after 10,000 bad answers reached users.

### Why This Happens

RAG systems are **fragile compound systems**:
- Embedding changes affect retrieval
- Prompt changes affect generation
- Threshold changes affect reranking
- **Small changes compound into large impacts**

### The Solution

**Automated regression testing in CI/CD**:
- Test every PR against quality metrics
- Block merges that degrade performance
- Version models with instant rollback
- Detect issues before production

**Key Metrics** (from script):
- Faithfulness: ≥0.75
- Answer Relevancy: ≥0.70
- Context Precision: ≥0.65
- P95 Latency: ≤2000ms
- Cost per Query: ≤$0.01

In [ ]:
# Setup: Import core module
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd()))

# Import our module
import l2_regression_testing_cicd as reg_test
import config

print("✓ Module 8.3: Regression Testing & CI/CD")
print(f"✓ Thresholds configured:")
print(f"  - Faithfulness: ≥{reg_test.RegressionMetrics.FAITHFULNESS_THRESHOLD}")
print(f"  - Relevancy: ≥{reg_test.RegressionMetrics.RELEVANCY_THRESHOLD}")
print(f"  - Precision: ≥{reg_test.RegressionMetrics.PRECISION_THRESHOLD}")

# Expected:
# ✓ Module 8.3: Regression Testing & CI/CD
# ✓ Thresholds configured:
#   - Faithfulness: ≥0.75
#   - Relevancy: ≥0.70
#   - Precision: ≥0.65

## Section 2: Prerequisites & Dependencies

### Required Modules
You must complete these before starting:
- **M3 (Deployment)**: Understand production deployment patterns
- **M8.1 (RAGAS Evaluation)**: Know how to measure RAG quality
- **M8.2 (A/B Testing)**: Understand comparison testing

### New Dependencies (from script)
```bash
pip install dvc==3.48.4
pip install pytest-benchmark==4.0.0
pip install dvc-s3==3.2.0
```

### What You'll Build
1. **Regression Test Suite**: pytest tests for quality metrics
2. **GitHub Actions Workflow**: Automated CI pipeline
3. **DVC Model Versioning**: Track models/embeddings/prompts
4. **Safe Deployment**: Canary testing with auto-rollback

In [ ]:
# Verify prerequisites and configuration
import json

# Check configuration
config_info = config.get_config_info()

print("✓ Configuration Validation:")
print(f"  - Test subset size: {config_info['test_config']['subset_size']} questions")
print(f"  - CI timeout: <{config_info['ci_cd']['timeout_minutes']} minutes")
print(f"  - Canary threshold: {config_info['ci_cd']['canary_pass_threshold']:.0%} pass rate")

# Check if services are available
validation = config.validate_config()
has_services = any(validation.values())
print(f"  - Services available: {has_services}")

# Expected:
# ✓ Configuration Validation:
#   - Test subset size: 50 questions
#   - CI timeout: <10 minutes
#   - Canary threshold: 80% pass rate
#   - Services available: True/False (depending on .env)

## Section 3: Theory Foundation - Why RAG CI/CD is Different

### Traditional Software Testing vs RAG Testing

**Traditional Software**:
- Deterministic: Same input → Same output
- Unit tests verify logic correctness
- Integration tests check component interaction

**RAG Systems**:
- Non-deterministic: LLMs produce varied outputs
- Must test **model behavior** not just code logic
- Need to catch **quality degradation** not just bugs
- Must validate **multiple metrics** (quality, speed, cost)

### The RAG CI/CD Workflow

```
PR Created → GitHub Actions Triggered
          ↓
    Pull DVC-tracked models
          ↓
    Run regression test suite (50 questions)
          ↓
    Measure: Faithfulness, Relevancy, Precision, Latency, Cost
          ↓
    Compare against thresholds
          ↓
    Pass? → Allow merge
    Fail? → Block merge + Post PR comment
```

### Trade-off: Speed vs Thoroughness

**Script highlights**:
- **Fast CI**: 50 questions, ~3 minutes (for quick feedback)
- **Full Eval**: 500 questions, ~20 minutes (run nightly)
- Trade-off: Accept some blind spots for developer velocity

In [ ]:
# Demonstrate the difference between deterministic and non-deterministic testing

# Example: Traditional test (deterministic)
def add_numbers(a, b):
    return a + b

assert add_numbers(2, 3) == 5  # Always passes
print("✓ Traditional test: Deterministic and reliable")

# Example: RAG test (non-deterministic)
# Multiple runs of same query may produce different scores
print("\n✓ RAG Testing Requirements:")
print("  1. Test model behavior, not just code logic")
print("  2. Validate quality metrics (faithfulness, relevancy, precision)")
print("  3. Check performance (latency, cost)")
print("  4. Handle non-determinism with multiple runs + tolerance")

# Expected:
# ✓ Traditional test: Deterministic and reliable
# ✓ RAG Testing Requirements:
#   1. Test model behavior, not just code logic
#   2. Validate quality metrics...
#   [etc]

## Section 4: Implementation - Step 1: Regression Test Suite

### Overview

Build a pytest-based regression test suite that:
1. Tests on 50-question subset (fast CI path)
2. Measures all 5 key metrics
3. Compares against thresholds
4. Fails if ANY metric regresses

### Test Data Structure

Each test case includes:
- `question`: User query
- `expected_answer`: Ground truth answer
- `contexts`: Expected retrieved documents
- `ground_truth`: Brief reference answer

### The RegressionTestSuite Class

**Key features**:
- Configurable subset size (50 for CI, 500 for nightly)
- Measures: faithfulness, relevancy, precision, latency, cost
- Calculates P95 latency (not average - catches tail latencies)
- Handles failures gracefully (doesn't crash entire suite)

In [ ]:
# Load test data and create regression test suite
import json

# Load example test data
test_data_path = Path("test_data/example_data.json")
if test_data_path.exists():
    with open(test_data_path, 'r') as f:
        data = json.load(f)
        test_questions = data['test_questions']
    print(f"✓ Loaded {len(test_questions)} test questions")
else:
    # Create minimal test data if file doesn't exist
    test_questions = [{'question': f'Test question {i}', 'expected_answer': 'Test answer', 'contexts': ['Test context']} for i in range(10)]
    print("⚠️ Using mock test data (example_data.json not found)")

# Create test suite with subset
test_suite = reg_test.RegressionTestSuite(test_questions, use_subset=True)
print(f"✓ Created test suite with {len(test_suite.test_data)} questions")
print(f"  - Use subset: True (fast CI mode)")
print(f"  - Full suite would use: {len(test_questions)} questions")

# Expected:
# ✓ Loaded 10 test questions
# ✓ Created test suite with 10 questions
#   - Use subset: True (fast CI mode)
#   - Full suite would use: 10 questions

## Section 5: Steps 2 & 3 - GitHub Actions Workflow & DVC Versioning

### Step 2: GitHub Actions Workflow

Typical `.github/workflows/regression-tests.yml`:
```yaml
name: RAG Regression Tests
on:
  pull_request:
  push:
    branches: [main, production]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Setup Python
        uses: actions/setup-python@v4
      - name: Install dependencies
        run: pip install -r requirements.txt
      - name: Pull DVC models
        run: dvc pull
      - name: Run regression tests
        run: pytest tests/ --benchmark-only
```

### Step 3: DVC Model Versioning

**Why DVC?**
- Track models/embeddings/prompts like code
- Store large files in S3 (not Git)
- Enable instant rollback to any version
- Team collaboration without conflicts

**Key Commands**:
```bash
dvc init                          # Initialize DVC
dvc remote add -d s3storage s3://bucket/path  # Configure S3
dvc add models/                   # Track models directory
git add models.dvc .gitignore     # Commit DVC metadata
git tag v1.0.0                    # Tag version
dvc push                          # Upload to S3
```

In [ ]:
# Demonstrate DVC version management
from pathlib import Path

# Create DVC version manager
models_dir = Path("models")
dvc_manager = reg_test.DVCVersionManager(models_dir=models_dir)

print("✓ DVC Version Manager initialized")
print(f"  - Models directory: {models_dir}")
print(f"  - DVC directory: .dvc")

# List available versions (will be empty if not using Git)
versions = dvc_manager.list_versions()
print(f"  - Available versions: {len(versions)}")

# Note: Actual DVC operations require Git repo with tags
print("\n⚠️ Note: Full DVC operations require:")
print("  1. Git repository initialized")
print("  2. DVC initialized (dvc init)")
print("  3. S3 remote configured")
print("  4. Model versions tagged")

# Expected:
# ✓ DVC Version Manager initialized
#   - Models directory: models
#   - DVC directory: .dvc
#   - Available versions: 0
# ⚠️ Note: Full DVC operations require...

## Section 6: Common Failures & Debugging

### Failure #1: CI Pipeline Too Slow (>10 minutes)

**Problem**: Full test suite takes 20+ minutes, developers context-switch  
**Solution**: Separate fast CI (50 questions) from nightly eval (500 questions)

```python
# pytest.ini configuration
[pytest]
markers =
    fast: Fast regression tests for CI (50 questions)
    full: Full evaluation suite (500 questions)

# Run in CI
pytest -m fast

# Run nightly
pytest -m full
```

### Failure #2: Flaky Tests (Intermittent Failures)

**Problem**: Single-run tests sensitive to infrastructure variance  
**Solution**: Run 5 times, use median, apply ±20% tolerance band

### Failure #3: Wrong Regression Thresholds

**Problem**: Too sensitive → blocks every PR (false positives)  
           Too loose → misses real regressions (false negatives)  
**Solution**: Calibrate using historical data: `threshold = baseline - 2*std`

### Failure #4: DVC Merge Conflicts

**Problem**: Simultaneous model changes → conflicting MD5 hashes  
**Solution**: Test both versions, choose better faithfulness score

### Failure #5: Rollback Failures (Previous Version Lost)

**Problem**: S3 lifecycle deleted old models before rollback needed  
**Solution**: Configure S3 to retain minimum 10 versions for 90 days

In [ ]:
# Demonstrate Flaky Test Handler and Threshold Calibrator

# 1. Flaky Test Handler
print("=" * 60)
print("Failure #2: Flaky Test Handling")
print("=" * 60)

flaky_handler = reg_test.FlakyTestHandler(num_runs=5, tolerance_pct=20.0)

# Mock flaky test function
import random
def mock_flaky_metric():
    # Simulates a metric that varies between 0.70-0.80
    return 0.75 + random.uniform(-0.05, 0.05)

# Run with stability
median, all_values = flaky_handler.run_stable_test(mock_flaky_metric)
print(f"✓ Median: {median:.3f} | All runs: {[f'{v:.3f}' for v in all_values]}")

# 2. Threshold Calibrator
print("\n" + "=" * 60)
print("Failure #3: Threshold Calibration")
print("=" * 60)

calibrator = reg_test.ThresholdCalibrator(Path("baseline_metrics_test.json"))

# Simulate adding baseline measurements
for i in range(15):
    calibrator.add_measurement('faithfulness', 0.78 + (i % 5) * 0.01)

# Calculate calibrated threshold
threshold = calibrator.calculate_threshold('faithfulness', num_std=2.0)
print(f"✓ Calibrated threshold: {threshold:.3f}")

# Expected output (abbreviated):
# ✓ Median: 0.750 | All runs: ['0.745', '0.752', ...]
# ✓ Calibrated threshold: 0.760

## Section 7: Decision Card - When to Use CI/CD (and When NOT)

### When CI/CD is the RIGHT Choice

✅ **USE when**:
- Deploy **20-100 times/month**
- Team of **3-10 engineers**
- Budget **≥$150/month**
- Have product-market fit (stable requirements)
- Need production confidence before each deploy
- Can dedicate 20% time for CI/CD maintenance

### When CI/CD is OVERKILL

❌ **AVOID when**:
- Deploy **<5 times/month** (manual testing more efficient)
- Team **<3 people** (maintenance overhead too high)
- Budget **<$50/month** (can't afford GitHub Actions + S3)
- **Pre-PMF** (requirements change too fast)
- Need **<100ms latency** (testing overhead unacceptable)
- Solo developer or 2-person team

### Alternative Approaches

| Approach | Best For | Cost | Setup Time |\n|----------|----------|------|------------|\n| Manual Testing | <3 people, <5 deploys/month | $0 | 0 hours |\n| Staged Rollouts | 3-10 people, 10-50 deploys/month | $50/mo | 8 hours |\n| GitHub Actions CI/CD | 3-10 people, 20-100 deploys/month | $150/mo | 16 hours |\n| Managed Platforms | 10+ people, 100+ deploys/month | $2000/mo | 80+ hours |

### Reality Check: What This DOESN'T Do

- ⚠️ **Doesn't catch all regressions** (50-question subset has blind spots)
- ⚠️ **Doesn't replace production monitoring** (edge cases slip through)
- ⚠️ **Adds complexity** (400+ lines of CI/CD code to maintain)
- ⚠️ **Slows PR velocity** (3-5 minute wait on every PR)

In [ ]:
# Use decision helper to determine if CI/CD is appropriate

print("=" * 70)
print("DECISION HELPER: Should You Use CI/CD?")
print("=" * 70)

scenarios = [
    {
        'name': 'Startup (2 people, 3 deploys/month)',
        'deploys': 3,
        'team': 2,
        'budget': 100,
        'pmf': False
    },
    {
        'name': 'Growing Team (5 people, 40 deploys/month)',
        'deploys': 40,
        'team': 5,
        'budget': 300,
        'pmf': True
    },
    {
        'name': 'Enterprise (20 people, 150 deploys/month)',
        'deploys': 150,
        'team': 20,
        'budget': 2000,
        'pmf': True
    }
]

for scenario in scenarios:
    print(f"\n{scenario['name']}:")
    should_use, reason = reg_test.should_use_cicd(
        scenario['deploys'],
        scenario['team'],
        scenario['budget'],
        scenario['pmf']
    )
    status = "✅ USE CI/CD" if should_use else "❌ AVOID CI/CD"
    print(f"  {status}")
    print(f"  Reason: {reason}")

# Expected:
# Startup (2 people, 3 deploys/month):
#   ❌ AVOID CI/CD
#   Reason: Deploy <5 times/month - manual testing more efficient
# Growing Team (5 people, 40 deploys/month):
#   ✅ USE CI/CD
#   Reason: Ideal fit...
# [etc]

## Section 8: Production Considerations & Cost Scaling

### Cost Breakdown (from script)

**Small Scale** (10 deploys/month):
- GitHub Actions: $20-30
- S3 Storage (DVC): $15-20
- API costs (testing): $15-30
- **Total: $50-80/month**

**Medium Scale** (50 deploys/month):
- GitHub Actions: $80-100
- S3 Storage: $30-40
- API costs: $40-60
- **Total: $150-200/month**

**Large Scale** (100+ deploys/month):
- GitHub Actions: $300-400
- S3 Storage: $80-100
- API costs: $120-300
- **Total: $500-800/month**
- Consider managed platform at this scale

### Monitoring Requirements

**CI Pipeline Health**:
- CI duration P95 **<8 minutes** (target <5 minutes)
- Test flakiness **<5%** (rerun rate)
- PR block rate **2-5%** (false positive sweet spot)

**Quality Metrics Tracking**:
- Track baseline metrics over time
- Alert on threshold degradation
- Review calibration quarterly

### Team Responsibilities

- **3-5 people**: Rotating "CI shepherd" role (1 week rotations)
- **6-10 people**: 20% MLOps time allocation (1 person part-time)
- **10+ people**: Full-time MLOps engineer

In [ ]:
# Cost estimation for different deployment scales

print("=" * 70)
print("CI/CD COST ESTIMATION")
print("=" * 70)

scales = [
    (10, 3, "Small"),
    (50, 5, "Medium"),
    (100, 10, "Large")
]

for deploys, team, scale_name in scales:
    print(f"\n{scale_name} Scale ({deploys} deploys/month, {team} engineers):")
    costs = reg_test.estimate_cicd_costs(deploys, team)
    print(f"  - GitHub Actions: ${costs['github_actions']:.2f}")
    print(f"  - S3 Storage:     ${costs['storage']:.2f}")
    print(f"  - API Testing:    ${costs['api_testing']:.2f}")
    print(f"  - TOTAL:          ${costs['total']:.2f}/month")

print("\n" + "=" * 70)
print("💡 Cost Optimization Tips:")
print("  - Use GPT-3.5 instead of GPT-4 for testing (3x cheaper)")
print("  - Cache embeddings to reduce API calls")
print("  - Run full eval nightly, not on every PR")
print("  - Use self-hosted runners if >100 deploys/month")

# Expected:
# Small Scale (10 deploys/month, 3 engineers):
#   - GitHub Actions: $0.00
#   - S3 Storage:     $2.30
#   - API Testing:    $1.50
#   - TOTAL:          $3.80/month
# [Cost breakdown for each scale]

## Section 9: Summary & Next Steps

### What You Learned

1. **Problem**: Minor RAG changes compound into 40% quality drops
2. **Solution**: Automated regression testing in CI/CD
3. **Implementation**:
   - Regression test suite (50 questions, ~3 minutes)
   - GitHub Actions workflow (auto-test every PR)
   - DVC model versioning (instant rollback)
   - Safe deployment with canary testing

4. **Common Failures & Fixes**:
   - Slow CI → Use subset testing
   - Flaky tests → Run 5x, use median
   - Wrong thresholds → Calibrate with baseline - 2*std
   - DVC conflicts → Test both, choose winner
   - Lost versions → S3 retention policy

5. **Decision Framework**:
   - **Use**: 20-100 deploys/month, 3-10 people, $150+ budget
   - **Avoid**: <5 deploys/month, <3 people, pre-PMF

### Key Takeaways

✅ **Do**:
- Test on 50-question subset for fast CI (<5 min)
- Version models with DVC for instant rollback
- Calibrate thresholds using historical data
- Monitor false positive rate (target 2-5%)

❌ **Don't**:
- Run full 500-question eval in CI (too slow)
- Use fixed thresholds (causes false positives/negatives)
- Deploy without rollback capability
- Implement CI/CD if deploying <5 times/month

### Next Steps

1. **Complete PractaThon Challenge**:
   - Easy (60 min): Basic workflow with 3 metrics
   - Medium (90-120 min): Add latency/cost, DVC versioning
   - Hard (4-5 hours): Production-grade with canary testing

2. **Proceed to M8.4**: Human-in-the-Loop Evaluation

In [ ]:
# Final summary and module completion check

print("=" * 70)
print("MODULE 8.3 COMPLETE ✓")
print("=" * 70)

print("\n📊 Configuration Summary:")
config_summary = config.get_config_info()
print(f"  - Test subset: {config_summary['test_config']['subset_size']} questions")
print(f"  - Faithfulness threshold: ≥{config_summary['thresholds']['faithfulness']}")
print(f"  - Answer relevancy threshold: ≥{config_summary['thresholds']['answer_relevancy']}")
print(f"  - Context precision threshold: ≥{config_summary['thresholds']['context_precision']}")
print(f"  - P95 latency threshold: ≤{config_summary['thresholds']['p95_latency_ms']}ms")
print(f"  - Cost per query threshold: ≤${config_summary['thresholds']['cost_per_query']}")

print("\n🎯 Key Components Covered:")
print("  ✓ Regression test suite")
print("  ✓ GitHub Actions workflow")
print("  ✓ DVC model versioning")
print("  ✓ Safe deployment with canary testing")
print("  ✓ Flaky test handling")
print("  ✓ Threshold calibration")
print("  ✓ Decision framework")
print("  ✓ Cost estimation")

print("\n🚀 Next Steps:")
print("  1. Complete PractaThon challenge (choose difficulty)")
print("  2. Set up GitHub Actions workflow in your repo")
print("  3. Initialize DVC and configure S3 remote")
print("  4. Proceed to M8.4: Human-in-the-Loop Evaluation")

print("\n" + "=" * 70)
print("Thank you for completing Module 8.3!")
print("=" * 70)

# Expected:
# MODULE 8.3 COMPLETE ✓
# [Configuration summary]
# [Key components covered]
# [Next steps]